In [ ]:
import os
from dotenv import load_dotenv

from future_rag.indexing.embeddings import EmbeddingFactory
from future_rag.indexing.indexer import IndexBuilder
from future_rag.knowledge.chunker import DocumentChunker
from future_rag.knowledge.loaders import DocumentLoader
from future_rag.llms.factory import LLMFactory
from future_rag.pipelines.basic_rag import BasicRAG
from future_rag.retrieval.retriever import Retriever


def main():
    # Load environment variables (DEEPSEEK_API_KEY, etc.) from .env
    load_dotenv()

    # 1. Load documents
    pdf_path = "../data/knowledge/pdf"
    print(f"Loading documents from {pdf_path}...")
    documents = DocumentLoader.load(pdf_path)

    # 2. Chunk documents into nodes
    print("Chunking documents...")
    chunker = DocumentChunker(chunk_size=1024, chunk_overlap=256)
    nodes = chunker.split(documents)

    # 3. Create embedding model
    print("Initializing embedding model...")
    embedding = EmbeddingFactory.create()

    # 4. Build index
    print("Building vector index...")
    builder = IndexBuilder(embedding)
    index = builder.build(nodes)

    # 5. Create retriever
    retriever = Retriever(index)

    # 6. Create LLM (registers in Settings.llm and returns the instance)
    print("Initializing LLM...")
    llm = LLMFactory.create(provider="deepseek")

    # 7. Create pipeline
    rag = BasicRAG(
        retriever=retriever,
        llm=llm,
    )

    # 8. Ask a question
    question = "What are the main principles of the FUTURE-AI guideline?"
    print(f"\nAsking: '{question}'\n")
    
    answer = rag.query(question)

    # 9. Display results
    print("=== ANSWER ===")
    # Safely handles both raw string return and LlamaIndex Response objects
    answer_text = getattr(answer, "response", str(answer))
    print(answer_text)

    # Print source citations if returned as a LlamaIndex Response object
    if hasattr(answer, "source_nodes") and answer.source_nodes:
        print("\n=== SOURCES ===")
        for idx, source in enumerate(answer.source_nodes, 1):
            score = getattr(source, "score", None)
            score_fmt = f" (Score: {score:.4f})" if score is not None else ""
            content = source.get_content() if hasattr(source, "get_content") else str(source)
            print(f"[{idx}]{score_fmt}\n{content[:200]}...\n")


if __name__ == "__main__":
    main()

2026-07-17 20:26:50,726 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 20:26:50,754 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-17 20:26:50,896 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 20:26:50,919 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-17 20:26:50,924 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-17 20:26:51,056 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

Based on the provided context, the main principles of the FUTURE-AI guideline are: fairness, universality, traceability, usability, robustness, and explainability.


In [1]:
from future_rag.knowledge.loaders import DocumentLoader
from future_rag.knowledge.chunker import DocumentChunker

from future_rag.indexing.embeddings import EmbeddingFactory
from future_rag.indexing.vectorstore import VectorStore
from future_rag.indexing.indexer import IndexBuilder

from future_rag.retrieval.retriever import Retriever

from future_rag.llms.factory import LLMFactory
from future_rag.pipelines.basic_rag import BasicRAG

# -------------------------------------------------------
# Load documents
# -------------------------------------------------------

documents = DocumentLoader.load("../data/knowledge/pdf")

print(f"Loaded {len(documents)} document(s)")

# -------------------------------------------------------
# Chunk documents
# -------------------------------------------------------

chunker = DocumentChunker(
    chunk_size=1024,
    chunk_overlap=256,
)

nodes = chunker.split(documents)

print(f"Created {len(nodes)} chunks")

# -------------------------------------------------------
# Create embedding model
# -------------------------------------------------------

embedding = EmbeddingFactory.create()

print("Embedding model loaded.")

# -------------------------------------------------------
# Connect to ChromaDB
# -------------------------------------------------------

vector_store = VectorStore().connect()

print("Connected to ChromaDB.")

# -------------------------------------------------------
# Build vector index
# -------------------------------------------------------

builder = IndexBuilder(
    embedding_model=embedding,
    vector_store=vector_store,
)

index = builder.build(nodes)

print("Index created successfully.")

# -------------------------------------------------------
# Create retriever
# -------------------------------------------------------

retriever = Retriever(index)

print("Retriever created.")

# -------------------------------------------------------
# Create LLM
# -------------------------------------------------------

llm = LLMFactory.create()

print("LLM loaded.")

# -------------------------------------------------------
# Create RAG pipeline
# -------------------------------------------------------

rag = BasicRAG(
    retriever=retriever,
    llm=llm,
)

print("RAG pipeline ready.")

# -------------------------------------------------------
# Ask a question
# -------------------------------------------------------

question = "What are the main principles of the FUTURE-AI guideline?"

answer = rag.query(question)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(answer)

d:\FUTURE AI Workspace\future-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 22 document(s)
Created 60 chunks


2026-07-18 00:40:13,947 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 00:40:13,949 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-18 00:40:13,966 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-18 00:40:14,105 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 00:40:14,126 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-18 00:40:14,143 - INFO - Loading SentenceTransformer model from BAAI/b

Embedding model loaded.
Connected to ChromaDB.
Index created successfully.
Retriever created.
LLM loaded.
RAG pipeline ready.


2026-07-18 00:41:01,086 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"



Question:
What are the main principles of the FUTURE-AI guideline?

Answer:
Based on the provided context, the main principles of the FUTURE-AI guideline are fairness, universality, traceability, usability, robustness, and explainability.


In [9]:
import importlib
import future_rag.indexing.vectorstore as vs

importlib.reload(vs)

print(vs.VectorStore)

<class 'future_rag.indexing.vectorstore.VectorStore'>
